# Russian sentiment: morphology-aware tokenisation and SHAP

Final experiment comparing:

1. standard SentencePiece BPE
2. semi-supervised Morfessor tokens with BPE fallback

Morfessor segments seen in the training corpus are kept whole. BPE is only used when a segment is not available in the morphology-aware vocabulary.

## 1. Setup

In [1]:
%pip install -q pandas numpy scikit-learn morfessor sentencepiece "transformers<5" accelerate shap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 118.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import os
import re
import copy
import pickle
import random
import difflib
from collections import Counter

import numpy as np
import pandas as pd
import morfessor
import sentencepiece as spm
import shap

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaConfig, RobertaForSequenceClassification

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

seed = 42

def set_seed(value):
    random.seed(value)
    np.random.seed(value)
    torch.manual_seed(value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(value)

set_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

results_dir = "results_final"
models_dir = "models_final"
os.makedirs(results_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)

print("device:", device)

device: cuda


## 2. RuSentiment data

In [3]:
random_posts = pd.read_csv("rusentiment_random_posts.csv")
preselected_posts = pd.read_csv("rusentiment_preselected_posts.csv")
test_data = pd.read_csv("rusentiment_test.csv")

labels = ["negative", "neutral", "positive", "speech"]

data = pd.concat([random_posts, preselected_posts], ignore_index=True)
data = data[data["label"].isin(labels)].copy()
test_data = test_data[test_data["label"].isin(labels)].copy()

for frame in [data, test_data]:
    frame.dropna(subset=["text", "label"], inplace=True)
    frame.drop(
        frame[frame["text"].astype(str).str.strip().eq("")].index,
        inplace=True
    )

duplicates = data["text"].duplicated().sum()
data = data.drop_duplicates(subset="text").reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

train_data, val_data = train_test_split(
    data,
    test_size=0.15,
    stratify=data["label"],
    random_state=seed
)

train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)

token_pattern = re.compile(
    r"[А-Яа-яЁё]+|[A-Za-z]+|\d+|[^\w\s]",
    re.UNICODE
)

def normalise_text(text):
    return " ".join(token_pattern.findall(str(text).lower()))

for frame in [train_data, val_data, test_data]:
    frame["normalised_text"] = frame["text"].apply(normalise_text)

print("duplicates removed:", duplicates)
print("train:", len(train_data))
print("validation:", len(val_data))
print("test:", len(test_data))

duplicates removed: 76
train: 20440
validation: 3608
test: 2621


## 3. Semi-supervised Morfessor

In [4]:
sig_train = pd.read_csv(
    "rus.word.train.tsv",
    sep="\t",
    header=None,
    names=["word", "segmentation", "code"]
)

sig_dev = pd.read_csv(
    "rus.word.dev.tsv",
    sep="\t",
    header=None,
    names=["word", "segmentation", "code"]
)

sig_test = pd.read_csv(
    "rus.word.test.gold.tsv",
    sep="\t",
    header=None,
    names=["word", "segmentation", "code"]
)

def parse_segments(segmentation):
    return [part.replace("@@", "") for part in str(segmentation).split()]

def surface_match(row):
    return "".join(parse_segments(row["segmentation"])) == str(row["word"])

sig_train = sig_train[sig_train.apply(surface_match, axis=1)].copy()
sig_dev = sig_dev[sig_dev.apply(surface_match, axis=1)].copy()
sig_test = sig_test[sig_test.apply(surface_match, axis=1)].copy()

russian_word_pattern = re.compile(r"[а-яё]+")
word_counts = Counter()

for text in train_data["normalised_text"]:
    word_counts.update(russian_word_pattern.findall(text))

training_words = [(count, word) for word, count in word_counts.items()]

gold_annotations = {
    str(row["word"]): [parse_segments(row["segmentation"])]
    for _, row in sig_train.iterrows()
}

morfessor_model = morfessor.BaselineModel()
morfessor_model.load_data(training_words)
morfessor_model.set_annotations(gold_annotations)
morfessor_model.train_batch(max_epochs=10)

100% (111114 of 111114) |################| Elapsed Time: 0:01:01 Time:  0:01:01
100% (111114 of 111114) |################| Elapsed Time: 0:00:42 Time:  0:00:42
100% (111114 of 111114) |################| Elapsed Time: 0:00:52 Time:  0:00:52
100% (111114 of 111114) |################| Elapsed Time: 0:00:44 Time:  0:00:44


(4, 12890767.074001476)

In [5]:
def boundaries(segments):
    result = set()
    position = 0
    for segment in segments[:-1]:
        position += len(segment)
        result.add(position)
    return result

def evaluate_morfessor(model, dataframe):
    tp = fp = fn = exact = 0
    errors = Counter()

    for _, row in dataframe.iterrows():
        word = str(row["word"])
        gold = parse_segments(row["segmentation"])
        predicted = list(model.viterbi_segment(word)[0])

        gold_b = boundaries(gold)
        pred_b = boundaries(predicted)

        tp += len(gold_b & pred_b)
        fp += len(pred_b - gold_b)
        fn += len(gold_b - pred_b)
        exact += int(predicted == gold)

        if pred_b == gold_b:
            errors["correct"] += 1
        elif gold_b < pred_b:
            errors["over-segmented"] += 1
        elif pred_b < gold_b:
            errors["under-segmented"] += 1
        else:
            errors["different boundaries"] += 1

    precision = tp / (tp + fp) if tp + fp else 1.0
    recall = tp / (tp + fn) if tp + fn else 1.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "exact_match": exact / len(dataframe)
    }, errors

dev_metrics, _ = evaluate_morfessor(morfessor_model, sig_dev)
test_morph_metrics, test_morph_errors = evaluate_morfessor(morfessor_model, sig_test)

morphology_results = pd.DataFrame([
    {"split": "dev", **dev_metrics},
    {"split": "test", **test_morph_metrics}
])

display(morphology_results.round(3))
print(pd.Series(test_morph_errors))

morphology_results.to_csv(
    f"{results_dir}/morphology_results.csv",
    index=False
)

,split,precision,recall,f1,exact_match
0,dev,0.808,0.967,0.880,0.858
1,test,0.797,0.966,0.874,0.852


correct                 7267
over-segmented           964
different boundaries     202
under-segmented           93
dtype: int64


## 4. Tokenizers

The morphology-aware tokenizer keeps every Cyrillic Morfessor segment observed in the RuSentiment training split as a whole token. A small BPE model supplies fallback pieces for material that is not in that training vocabulary. The standard BPE tokenizer is then trained with the same final vocabulary size as the morphology-aware tokenizer.

In [6]:
segment_cache = {}

def segment_word(word):
    if word not in segment_cache:
        segment_cache[word] = list(morfessor_model.viterbi_segment(word)[0])
    return segment_cache[word]

def morfessor_segment_text(text):
    parts = []
    for token in token_pattern.findall(text):
        if re.fullmatch(r"[а-яё]+", token):
            parts.extend(segment_word(token))
        else:
            parts.append(token)
    return " ".join(parts)

for frame in [train_data, val_data, test_data]:
    frame["morfessor_text"] = frame["normalised_text"].apply(morfessor_segment_text)

morph_counts = Counter()
for text in train_data["morfessor_text"]:
    for part in text.split():
        if re.fullmatch(r"[а-яё]+", part):
            morph_counts[part] += 1

morph_vocabulary = set(morph_counts)
print("training Morfessor segments:", len(morph_vocabulary))

training Morfessor segments: 18105


In [7]:
fallback_vocab_size = 3000

with open("fallback_bpe_train.txt", "w", encoding="utf-8") as f:
    for text in train_data["normalised_text"]:
        f.write(text + "\n")

spm.SentencePieceTrainer.train(
    input="fallback_bpe_train.txt",
    model_prefix="fallback_bpe",
    vocab_size=fallback_vocab_size,
    model_type="bpe",
    character_coverage=1.0,
    split_by_whitespace=True,
    hard_vocab_limit=False,
    unk_id=0,
    pad_id=1,
    bos_id=2,
    eos_id=3
)

fallback_bpe = spm.SentencePieceProcessor(
    model_file="fallback_bpe.model"
)

class MorphologyFallbackTokenizer:
    def __init__(self, morphs, fallback):
        self.fallback = fallback
        self.specials = ["<unk>", "<pad>", "<s>", "</s>"]

        fallback_pieces = [
            fallback.id_to_piece(i)
            for i in range(fallback.vocab_size())
            if fallback.id_to_piece(i) not in self.specials
        ]

        pieces = self.specials + sorted(morphs)
        seen = set(pieces)

        for piece in fallback_pieces:
            if piece not in seen:
                pieces.append(piece)
                seen.add(piece)

        self.pieces = pieces
        self.piece_to_id = {piece: i for i, piece in enumerate(pieces)}
        self.morphs = set(morphs)

    def encode(self, text, out_type=int):
        ids = []

        for part in str(text).split():
            if part in self.morphs:
                ids.append(self.piece_to_id[part])
            else:
                for piece in self.fallback.encode(part, out_type=str):
                    ids.append(self.piece_to_id.get(piece, self.unk_id()))

        if out_type == str:
            return [self.id_to_piece(i) for i in ids]
        return ids

    def id_to_piece(self, token_id):
        return self.pieces[token_id]

    def vocab_size(self):
        return len(self.pieces)

    def unk_id(self):
        return 0

    def pad_id(self):
        return 1

    def bos_id(self):
        return 2

    def eos_id(self):
        return 3

morphology_bpe = MorphologyFallbackTokenizer(
    morph_vocabulary,
    fallback_bpe
)

matched_vocab_size = morphology_bpe.vocab_size()

with open("standard_bpe_train.txt", "w", encoding="utf-8") as f:
    for text in train_data["normalised_text"]:
        f.write(text + "\n")

spm.SentencePieceTrainer.train(
    input="standard_bpe_train.txt",
    model_prefix="standard_bpe_final",
    vocab_size=matched_vocab_size,
    model_type="bpe",
    character_coverage=1.0,
    split_by_whitespace=True,
    hard_vocab_limit=False,
    unk_id=0,
    pad_id=1,
    bos_id=2,
    eos_id=3
)

standard_bpe = spm.SentencePieceProcessor(
    model_file="standard_bpe_final.model"
)

print("standard BPE vocab:", standard_bpe.vocab_size())
print("morphology-aware vocab:", morphology_bpe.vocab_size())

standard BPE vocab: 20626
morphology-aware vocab: 20626


## 5. Tokenizer check

In [8]:
def clean_piece(piece):
    return piece.replace("▁", "")

examples = [
    "я очень недовольна этим",
    "это полное разочарование",
    "ужасный день",
    "какие милые люди"
]

for text in examples:
    normal = normalise_text(text)
    morph = morfessor_segment_text(normal)

    standard_tokens = [
        clean_piece(x)
        for x in standard_bpe.encode(normal, out_type=str)
    ]

    morphology_tokens = [
        clean_piece(x)
        for x in morphology_bpe.encode(morph, out_type=str)
    ]

    print("\n", text)
    print("standard: ", " | ".join(standard_tokens))
    print("Morfessor:", " | ".join(morph.split()))
    print("hybrid:   ", " | ".join(morphology_tokens))


 я очень недовольна этим
standard:  я | очень | недоволь | на | этим
Morfessor: я | очень | не | довольна | этим
hybrid:    я | очень | не | довольна | этим

 это полное разочарование
standard:  это | полное | разочарова | ние
Morfessor: это | полное | раз | очарование
hybrid:    это | полное | раз | очарование

 ужасный день
standard:  ужасный | день
Morfessor: ужас | ный | день
hybrid:    ужас | ный | день

 какие милые люди
standard:  какие | милые | люди
Morfessor: как | и | е | милые | люди
hybrid:    как | и | е | милые | люди


In [9]:
def unknown_rate(dataframe, text_column, tokenizer):
    total = unknown = 0

    for text in dataframe[text_column]:
        ids = tokenizer.encode(text, out_type=int)
        total += len(ids)
        unknown += sum(i == tokenizer.unk_id() for i in ids)

    return 100 * unknown / total if total else 0.0

def fallback_rate(dataframe):
    total = fallback = 0

    for text in dataframe["morfessor_text"]:
        for part in text.split():
            if re.fullmatch(r"[а-яё]+", part):
                total += 1
                fallback += int(part not in morphology_bpe.morphs)

    return 100 * fallback / total if total else 0.0

max_length = 128
rows = []

for split_name, frame in [
    ("train", train_data),
    ("validation", val_data),
    ("test", test_data)
]:
    standard_lengths = frame["normalised_text"].apply(
        lambda x: len(standard_bpe.encode(x, out_type=int))
    )
    morphology_lengths = frame["morfessor_text"].apply(
        lambda x: len(morphology_bpe.encode(x, out_type=int))
    )

    rows.extend([
        {
            "split": split_name,
            "condition": "standard BPE",
            "mean_tokens": standard_lengths.mean(),
            "percent_over_128": 100 * (standard_lengths > 128).mean(),
            "unk_percent": unknown_rate(frame, "normalised_text", standard_bpe),
            "fallback_percent": 0.0
        },
        {
            "split": split_name,
            "condition": "Morfessor + BPE fallback",
            "mean_tokens": morphology_lengths.mean(),
            "percent_over_128": 100 * (morphology_lengths > 128).mean(),
            "unk_percent": unknown_rate(frame, "morfessor_text", morphology_bpe),
            "fallback_percent": fallback_rate(frame)
        }
    ])

tokenizer_results = pd.DataFrame(rows)
display(tokenizer_results.round(3))

tokenizer_results.to_csv(
    f"{results_dir}/tokenizer_results.csv",
    index=False
)

,split,condition,mean_tokens,percent_over_128,unk_percent,fallback_percent
0,train,standard BPE,21.912,1.707,0.001,0.000
1,train,Morfessor + BPE fallback,23.185,1.952,0.001,0.000
2,validation,standard BPE,22.437,1.718,0.089,0.000
3,validation,Morfessor + BPE fallback,24.213,2.162,0.082,0.490
4,test,standard BPE,21.735,1.793,0.102,0.000
5,test,Morfessor + BPE fallback,23.367,2.098,0.095,0.462


## 6. Matched RoBERTa classifiers

In [10]:
label_names = ["negative", "neutral", "positive", "speech"]
label_to_id = {label: i for i, label in enumerate(label_names)}

for frame in [train_data, val_data, test_data]:
    frame["label_id"] = frame["label"].map(label_to_id)

class SentimentDataset(Dataset):
    def __init__(self, dataframe, text_column, tokenizer, max_length=128):
        self.dataframe = dataframe.reset_index(drop=True)
        self.text_column = text_column
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        token_ids = self.tokenizer.encode(
            row[self.text_column],
            out_type=int
        )[:self.max_length - 2]

        ids = [self.tokenizer.bos_id(), *token_ids, self.tokenizer.eos_id()]
        attention = [1] * len(ids)

        padding = self.max_length - len(ids)
        ids += [self.tokenizer.pad_id()] * padding
        attention += [0] * padding

        return {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention, dtype=torch.long),
            "labels": torch.tensor(int(row["label_id"]), dtype=torch.long)
        }

conditions = {
    "standard_bpe": {
        "name": "standard BPE",
        "tokenizer": standard_bpe,
        "text_column": "normalised_text"
    },
    "morphology_bpe": {
        "name": "Morfessor + BPE fallback",
        "tokenizer": morphology_bpe,
        "text_column": "morfessor_text"
    }
}

batch_size = 32
epochs = 8
learning_rate = 3e-4

def create_model(tokenizer):
    config = RobertaConfig(
        vocab_size=tokenizer.vocab_size(),
        hidden_size=96,
        num_hidden_layers=2,
        num_attention_heads=4,
        intermediate_size=192,
        max_position_embeddings=130,
        num_labels=len(label_names),
        unk_token_id=tokenizer.unk_id(),
        pad_token_id=tokenizer.pad_id(),
        bos_token_id=tokenizer.bos_id(),
        eos_token_id=tokenizer.eos_id()
    )
    return RobertaForSequenceClassification(config)

def evaluate_loader(model, loader):
    model.eval()
    true_labels = []
    predictions = []

    with torch.no_grad():
        for batch in loader:
            logits = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device)
            ).logits

            predictions.extend(logits.argmax(dim=1).cpu().numpy())
            true_labels.extend(batch["labels"].numpy())

    return (
        accuracy_score(true_labels, predictions),
        f1_score(true_labels, predictions, average="macro")
    )

def train_condition(key):
    config = conditions[key]
    set_seed(seed)

    train_set = SentimentDataset(
        train_data,
        config["text_column"],
        config["tokenizer"],
        max_length
    )
    val_set = SentimentDataset(
        val_data,
        config["text_column"],
        config["tokenizer"],
        max_length
    )

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)

    model = create_model(config["tokenizer"]).to(device)
    optimiser = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    best_f1 = -1
    best_state = None
    history = []

    print("\n" + config["name"])

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0

        for batch in train_loader:
            optimiser.zero_grad()

            output = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                labels=batch["labels"].to(device)
            )

            output.loss.backward()
            optimiser.step()
            total_loss += output.loss.item()

        accuracy, macro_f1 = evaluate_loader(model, val_loader)
        loss = total_loss / len(train_loader)

        history.append({
            "epoch": epoch,
            "loss": loss,
            "accuracy": accuracy,
            "macro_f1": macro_f1
        })

        print(
            epoch,
            "loss", round(loss, 4),
            "accuracy", round(accuracy, 4),
            "macro F1", round(macro_f1, 4)
        )

        if macro_f1 > best_f1:
            best_f1 = macro_f1
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    model.save_pretrained(f"{models_dir}/{key}")

    history = pd.DataFrame(history)
    history.to_csv(f"{results_dir}/{key}_history.csv", index=False)

    return model, history

models = {}
histories = {}

for key in conditions:
    models[key], histories[key] = train_condition(key)


standard BPE
1 loss 0.9538 accuracy 0.6829 macro F1 0.6053
2 loss 0.5938 accuracy 0.7242 macro F1 0.71
3 loss 0.3143 accuracy 0.712 macro F1 0.696
4 loss 0.1716 accuracy 0.709 macro F1 0.6997
5 loss 0.105 accuracy 0.6935 macro F1 0.6981
6 loss 0.071 accuracy 0.7065 macro F1 0.6848
7 loss 0.0555 accuracy 0.7015 macro F1 0.6935
8 loss 0.0423 accuracy 0.6924 macro F1 0.6925

Morfessor + BPE fallback
1 loss 0.9485 accuracy 0.7076 macro F1 0.6456
2 loss 0.5737 accuracy 0.7336 macro F1 0.7204
3 loss 0.3364 accuracy 0.7209 macro F1 0.7134
4 loss 0.2018 accuracy 0.7084 macro F1 0.7036
5 loss 0.1349 accuracy 0.6915 macro F1 0.6875
6 loss 0.0976 accuracy 0.7015 macro F1 0.6893
7 loss 0.0762 accuracy 0.7037 macro F1 0.6841
8 loss 0.0647 accuracy 0.7068 macro F1 0.6991


## 7. Classification results

In [11]:
def get_predictions(model, dataframe, text_column, tokenizer):
    dataset = SentimentDataset(
        dataframe,
        text_column,
        tokenizer,
        max_length
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    true_labels = []
    predictions = []
    probabilities = []

    model.eval()

    with torch.no_grad():
        for batch in loader:
            logits = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device)
            ).logits

            probs = torch.softmax(logits, dim=1)
            true_labels.extend(batch["labels"].numpy())
            predictions.extend(probs.argmax(dim=1).cpu().numpy())
            probabilities.extend(probs.cpu().numpy())

    return (
        np.array(true_labels),
        np.array(predictions),
        np.array(probabilities)
    )

rows = []
prediction_store = {}

for split_name, frame in [("validation", val_data), ("test", test_data)]:
    for key, config in conditions.items():
        true, pred, probs = get_predictions(
            models[key],
            frame,
            config["text_column"],
            config["tokenizer"]
        )

        prediction_store[(split_name, key)] = (true, pred, probs)

        rows.append({
            "split": split_name,
            "condition": config["name"],
            "accuracy": accuracy_score(true, pred),
            "macro_f1": f1_score(true, pred, average="macro")
        })

classification_results = pd.DataFrame(rows)
display(classification_results.round(4))

classification_results.to_csv(
    f"{results_dir}/classification_results.csv",
    index=False
)

for key, config in conditions.items():
    true, pred, _ = prediction_store[("test", key)]

    report = pd.DataFrame(
        classification_report(
            true,
            pred,
            target_names=label_names,
            output_dict=True,
            zero_division=0
        )
    ).transpose()

    report.to_csv(f"{results_dir}/{key}_test_report.csv")

,split,condition,accuracy,macro_f1
0,validation,standard BPE,0.7242,0.7100
1,validation,Morfessor + BPE fallback,0.7336,0.7204
2,test,standard BPE,0.7741,0.7353
3,test,Morfessor + BPE fallback,0.7669,0.7243


## 8. Select real examples

In [12]:
def cyrillic_ratio(text):
    visible = [c for c in str(text) if not c.isspace()]
    if not visible:
        return 0.0
    return len(re.findall(r"[А-Яа-яЁё]", str(text))) / len(visible)

def text_for_condition(text, key):
    normal = normalise_text(text)
    if key == "morphology_bpe":
        return morfessor_segment_text(normal)
    return normal

def predict_one(text, key):
    config = conditions[key]
    tokenizer = config["tokenizer"]
    model = models[key]

    clean = text_for_condition(text, key)
    token_ids = tokenizer.encode(clean, out_type=int)[:max_length - 2]

    ids = [tokenizer.bos_id(), *token_ids, tokenizer.eos_id()]
    attention = [1] * len(ids)

    padding = max_length - len(ids)
    ids += [tokenizer.pad_id()] * padding
    attention += [0] * padding

    input_ids = torch.tensor([ids], dtype=torch.long, device=device)
    attention_mask = torch.tensor([attention], dtype=torch.long, device=device)

    model.eval()
    with torch.no_grad():
        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).logits

    probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
    return int(probs.argmax()), probs

def display_tokens(text, key):
    clean = text_for_condition(text, key)
    tokenizer = conditions[key]["tokenizer"]

    return [
        clean_piece(tokenizer.id_to_piece(i))
        for i in tokenizer.encode(clean, out_type=int)[:max_length - 2]
    ]

candidate_rows = []

for index, row in val_data.iterrows():
    if row["label"] not in ["negative", "positive"]:
        continue
    if cyrillic_ratio(row["text"]) < 0.60:
        continue

    true_id = int(row["label_id"])
    standard_pred, standard_probs = predict_one(row["text"], "standard_bpe")
    morph_pred, morph_probs = predict_one(row["text"], "morphology_bpe")

    if standard_pred != true_id or morph_pred != true_id:
        continue

    confidence = min(
        standard_probs[true_id],
        morph_probs[true_id]
    )
    if confidence < 0.60:
        continue

    standard_tokens = display_tokens(row["text"], "standard_bpe")
    morph_tokens = display_tokens(row["text"], "morphology_bpe")

    if not (5 <= len(standard_tokens) <= 25 and 5 <= len(morph_tokens) <= 25):
        continue

    difference = 1 - difflib.SequenceMatcher(
        None,
        standard_tokens,
        morph_tokens
    ).ratio()

    if difference < 0.10:
        continue

    candidate_rows.append({
        "original_index": index,
        "text": row["text"],
        "true_label": row["label"],
        "true_id": true_id,
        "standard_confidence": standard_probs[true_id],
        "morphology_confidence": morph_probs[true_id],
        "tokenisation_difference": difference
    })

candidates = pd.DataFrame(candidate_rows)

selected = []
for label in ["negative", "positive"]:
    subset = candidates[candidates["true_label"] == label]
    selected.append(
        subset.sample(
            n=min(3, len(subset)),
            random_state=seed
        )
    )

selected_examples = pd.concat(selected, ignore_index=True)
selected_examples["example_id"] = np.arange(len(selected_examples)) + 1

display(
    selected_examples[
        [
            "example_id",
            "true_label",
            "text",
            "standard_confidence",
            "morphology_confidence"
        ]
    ]
)

selected_examples.to_csv(
    f"{results_dir}/selected_examples.csv",
    index=False
)

,example_id,true_label,text,standard_confidence,morphology_confidence
0,1,negative,"сукаааа, где мой комп!!!!",0.877559,0.891076
1,2,negative,"Жалко,что под стрыкало не делают дабстеп ремик...",0.751477,0.864600
2,3,negative,"я не хочу отношении,они заебали. когда ты люби...",0.756752,0.791688
3,4,positive,Лёжа в ванной хорошо думать о политике =),0.646872,0.624738
4,5,positive,А ты меня не перестаёшь удивлять..аахахаахах)))),0.882980,0.914570
5,6,positive,"песня с этого фильма,очень класный фильм,реком...",0.769784,0.923346


## 9. Morphology-focused SHAP

In [24]:
# Selected SHAP examples

chosen_indices = [686, 1713, 1253, 633, 840, 2848]

selected_examples = val_data.loc[chosen_indices].copy()

selected_examples["true_id"] = selected_examples["label_id"]
selected_examples["true_label"] = selected_examples["label"]
selected_examples["example_id"] = range(1, len(selected_examples) + 1)

display(
    selected_examples[
        ["example_id", "true_label", "text"]
    ]
)

# Morphology-focused SHAP

def is_cyrillic_piece(piece):
    return bool(re.search(r"[а-яё]", clean_piece(piece), re.IGNORECASE))


def prepare_shap(text, key):
    config = conditions[key]
    tokenizer = config["tokenizer"]
    model = models[key]

    clean = text_for_condition(text, key)
    token_ids = tokenizer.encode(clean, out_type=int)[:max_length - 2]
    pieces = [tokenizer.id_to_piece(i) for i in token_ids]

    feature_positions = [
        i for i, piece in enumerate(pieces)
        if is_cyrillic_piece(piece)
    ]

    feature_tokens = [
        clean_piece(pieces[i])
        for i in feature_positions
    ]

    def predict_masks(masks):
        masks = np.asarray(masks)

        if masks.ndim == 1:
            masks = masks.reshape(1, -1)

        all_ids = []
        all_attention = []

        for mask in masks:
            lookup = dict(zip(feature_positions, mask))

            ids = [tokenizer.bos_id()]
            attention = [1]

            for position, token_id in enumerate(token_ids):
                if position not in lookup or lookup[position] >= 0.5:
                    ids.append(token_id)
                    attention.append(1)
                else:
                    ids.append(tokenizer.pad_id())
                    attention.append(0)

            ids.append(tokenizer.eos_id())
            attention.append(1)

            padding = max_length - len(ids)

            ids += [tokenizer.pad_id()] * padding
            attention += [0] * padding

            all_ids.append(ids)
            all_attention.append(attention)

        probabilities = []

        model.eval()

        with torch.no_grad():
            for start in range(0, len(all_ids), 128):
                batch_ids = torch.tensor(
                    all_ids[start:start + 128],
                    dtype=torch.long,
                    device=device
                )

                batch_attention = torch.tensor(
                    all_attention[start:start + 128],
                    dtype=torch.long,
                    device=device
                )

                logits = model(
                    input_ids=batch_ids,
                    attention_mask=batch_attention
                ).logits

                probabilities.append(
                    torch.softmax(logits, dim=1).cpu().numpy()
                )

        return np.vstack(probabilities)

    return feature_tokens, predict_masks


def calculate_shap(text, class_id, key, nsamples=200):
    tokens, predictor = prepare_shap(text, key)

    if not tokens:
        return pd.DataFrame(
            columns=["token_index", "token", "shap_value"]
        )

    background = np.zeros((1, len(tokens)))
    full_mask = np.ones((1, len(tokens)))

    explainer = shap.KernelExplainer(
        lambda masks: predictor(masks)[:, class_id],
        background
    )

    values = np.asarray(
        explainer.shap_values(
            full_mask,
            nsamples=nsamples
        )
    ).reshape(-1)

    return pd.DataFrame({
        "token_index": np.arange(len(tokens)),
        "token": tokens,
        "shap_value": values
    })


all_shap = []

for _, row in selected_examples.iterrows():

    for key, config in conditions.items():

        result = calculate_shap(
            row["text"],
            int(row["true_id"]),
            key,
            nsamples=200
        )

        result["example_id"] = int(row["example_id"])
        result["condition"] = config["name"]
        result["true_label"] = row["true_label"]
        result["text"] = row["text"]

        all_shap.append(result)


shap_results = pd.concat(
    all_shap,
    ignore_index=True
)

shap_results["absolute_shap"] = (
    shap_results["shap_value"].abs()
)

print("SHAP complete")

,example_id,true_label,text
686,1,negative,"девушка может плакать по-разному. навзрыд, ист..."
1713,2,positive,"Народ! мы лучшие =) как бы пошло не звучало, н..."
1253,3,positive,Я люблю фармацевтов...о как сильно!!!!!
633,4,negative,А он заставлял меня ревновать...каждый день......
840,5,negative,А я не знаю даже сколько в моей душе ран и тр...
2848,6,positive,Все-таки старинная музыка находит удивительный...


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

SHAP complete


## 10. Final examples

In [14]:
def top_shap(example_id, condition, n=5):
    part = shap_results[
        (shap_results["example_id"] == example_id)
        & (shap_results["condition"] == condition)
    ].nlargest(n, "absolute_shap")

    return ", ".join(
        f"{row.token} ({row.shap_value:+.3f})"
        for _, row in part.iterrows()
    )

for _, row in selected_examples.iterrows():
    example_id = int(row["example_id"])

    print("\n" + "=" * 60)
    print("example", example_id, "-", row["true_label"])
    print(row["text"])

    print("\nstandard BPE")
    print("tokens:", " | ".join(display_tokens(row["text"], "standard_bpe")))
    print("SHAP:", top_shap(example_id, "standard BPE"))

    print("\nMorfessor + BPE fallback")
    print("tokens:", " | ".join(display_tokens(row["text"], "morphology_bpe")))
    print("SHAP:", top_shap(example_id, "Morfessor + BPE fallback"))


example 1 - negative
сукаааа, где мой комп!!!!

standard BPE
tokens: сука | ааа | , | где | мой | комп | ! | ! | ! | !
SHAP: сука (+0.738), комп (+0.069), где (+0.055), мой (-0.022), ааа (-0.019)

Morfessor + BPE fallback
tokens: сука | а | а | а | , | где | мой | комп | ! | ! | ! | !
SHAP: сука (+0.772), где (+0.049), а (+0.016), а (+0.016), а (+0.016)

example 2 - negative
Жалко,что под стрыкало не делают дабстеп ремиксы((

standard BPE
tokens: жалко | , | что | под | ст | ры | ка | ло | не | делают | дабсте | п | рем | ик | сы | ( | (
SHAP: ст (-0.168), жалко (+0.105), ик (-0.034), делают (-0.028), ло (+0.025)

Morfessor + BPE fallback
tokens: жалко | , | что | под | стр | ы | ка | ло | не | делают | дабстеп | ремикс | ы | ( | (
SHAP: жалко (+0.060), делают (-0.041), дабстеп (-0.025), стр (+0.020), ы (-0.019)

example 3 - negative
я не хочу отношении,они заебали. когда ты любишь а тебя наебали...

standard BPE
tokens: я | не | хочу | отно | шении | , | они | заебали | . | когда | т

In [25]:
# Display SHAP examples

def top_shap(example_id, condition, n=5):
    part = shap_results[
        (shap_results["example_id"] == example_id)
        & (shap_results["condition"] == condition)
    ].nlargest(n, "absolute_shap")

    return ", ".join(
        f"{row.token} ({row.shap_value:+.3f})"
        for _, row in part.iterrows()
    )


for _, row in selected_examples.iterrows():

    example_id = int(row["example_id"])

    print("\n" + "=" * 60)
    print("Example", example_id, "-", row["true_label"])
    print(row["text"])

    print("\nStandard BPE")
    print(
        "tokens:",
        " | ".join(
            display_tokens(
                row["text"],
                "standard_bpe"
            )
        )
    )
    print(
        "SHAP:",
        top_shap(
            example_id,
            "standard BPE"
        )
    )

    print("\nMorfessor + BPE fallback")
    print(
        "tokens:",
        " | ".join(
            display_tokens(
                row["text"],
                "morphology_bpe"
            )
        )
    )
    print(
        "SHAP:",
        top_shap(
            example_id,
            "Morfessor + BPE fallback"
        )
    )


Example 1 - negative
девушка может плакать по-разному. навзрыд, истерично, всхлипывая, ругаясь. но самый пиздец - это тихие слезы с каменным лицом и молчание.

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

Standard BPE
tokens: девушка | может | плакать | по | - | разному | . | нав | з | ры | д | , | истери | чно | , | вс | х | ли | пы | вая | , | ру | гая | сь | . | но | самый | пиздец | - | это | ти | хие | слезы | с | ка | мен | ным | лицом | и | молчание | . | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | ! | !
SHAP: пиздец (+0.189), слезы (+0.050), это (+0.031), истери (+0.029), плакать (+0.025)

Morfessor + BPE fallback
tokens: девушка | может | плакать | по | - | раз | ному | . | на | взрыд | , | истерично | , | вс | х | ли | п | ывая | , | руг | аясь | . | но | самый | пиздец | - | это | тих | и | е | слезы | с | ка | мен | ным | лицом | и | молчание | . | ! | ! | ! | ! | ! | !

In [27]:

summary_rows = []

for _, row in selected_examples.iterrows():
    example_id = int(row["example_id"])

    for key, condition_name in [
        ("standard_bpe", "Standard BPE"),
        ("morphology_bpe", "Morfessor + BPE fallback")
    ]:
        summary_rows.append({
            "Example": example_id,
            "Label": row["true_label"],
            "Condition": condition_name,
            "Tokens": " | ".join(
                display_tokens(row["text"], key)
            ),
            "Top SHAP features": top_shap(
                example_id,
                condition_name
            )
        })

shap_summary = pd.DataFrame(summary_rows)

display(shap_summary)

,Example,Label,Condition,Tokens,Top SHAP features
0,1,negative,Standard BPE,девушка | может | плакать | по | - | разному |...,
1,1,negative,Morfessor + BPE fallback,девушка | может | плакать | по | - | раз | ном...,"пиздец (+0.426), слезы (+0.051), плакать (+0.0..."
2,2,positive,Standard BPE,народ | ! | мы | лучшие | = | ) | как | бы | п...,
3,2,positive,Morfessor + BPE fallback,народ | ! | мы | лучшие | = | ) | как | бы | п...,"народ (-0.078), как (-0.046), лучшие (+0.043),..."
4,3,positive,Standard BPE,я | люблю | фа | рма | цев | тов | . | . | . |...,
5,3,positive,Morfessor + BPE fallback,я | люблю | фа | рма | цев | т | ов | . | . | ...,"люблю (+0.407), сильно (+0.173), ов (-0.092), ..."
6,4,negative,Standard BPE,а | он | заставля | л | меня | ре | вно | вать...,
7,4,negative,Morfessor + BPE fallback,а | он | за | ставлял | меня | ревновать | . |...,"ставлял (-0.108), больно (+0.067), а (-0.022),..."
8,5,negative,Standard BPE,а | я | не | знаю | даже | сколько | в | моей ...,
9,5,negative,Morfessor + BPE fallback,а | я | не | знаю | даже | сколько | в | моей ...,"трещин (-0.005), оскол (+0.004), знаю (-0.003)..."
